# PyG 로 GCN 만들고 학습하기 직접 다시 짜기

2주차 노트북의 PyTorch Geometric 부분을 직접 짜요. 그래프를 Data 상자에 담고, GCNConv 두 층 모델을 만들고, 학습 함수와 정확도, 그리고 MessagePassing 틀로 만든 GCN 층까지. w2-scratchgcn 에서 손으로 짠 GCN 과 같은 일을 PyG 가 어떻게 줄여 주는지 봐요. Colab 에서 열리고 torch_geometric 설치 칸이 앞에 붙어요.

**하는 법**
1. 위 메뉴 **런타임 > 런타임 유형 변경** 은 CPU 그대로 두어도 돼요.
2. 문제마다 **내 코드 칸**을 채우고 실행(Shift+Enter)한 뒤, 바로 아래 **채점 칸**을 실행해요.
3. `통과!` 가 나오면 성공, 빨간 `AssertionError` 가 나오면 마지막 줄의 한국어 안내를 읽고 고쳐요.
4. 막히면 **힌트**를 한 단계씩 펼쳐 보고, 그래도 안 되면 **정답 보기**를 펼쳐요.

## 준비: 필요한 도구 설치 (처음 한 번만 실행)

In [ ]:
!pip install -q torch_geometric

## 1. 3명짜리 그래프를 Data 로  (따라 치기)

노드 3개, 선 2개(양방향이라 4줄)인 그래프를 PyG 의 Data 상자에 담고 노드 수, 엣지 수를 확인해요.

- PyG(PyTorch Geometric)는 torch 위에서 그래프 신경망을 쉽게 짜게 해 주는 도구 모음이에요. 그래프 하나를 `Data` 라는 상자에 담아요.
- `edge_index` 는 위 줄 출발, 아래 줄 도착인 2 x (엣지 수) 표예요. 0-1, 1-2 친구 관계를 양방향으로 적어서 4열이에요 (c08).
- `dtype=torch.long` 은 정수(노드 번호), `dtype=torch.float` 는 실수(특징)예요 (c09).
- `Data(x=x, edge_index=edge_index)` 는 이름을 붙여 넣어요. 그러면 `simple_graph.x`, `simple_graph.num_nodes` 처럼 꺼내 써요.
- 원본 셀은 표를 여러 줄에 나눠 적었는데, 여기서는 한 줄로 붙였어요. 같은 코드예요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
from torch_geometric.data import Data

edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
x = torch.tensor([[-1.0], [0.0], [1.0]], dtype=torch.float)

simple_graph = Data(x=x, edge_index=edge_index)
print(simple_graph)
print("Number of nodes:", simple_graph.num_nodes)
print("Number of edges:", simple_graph.num_edges)
```

<details><summary>힌트 1</summary>

from torch_geometric.data import Data

</details>

<details><summary>힌트 2</summary>

edge_index 는 2줄짜리 정수 표

</details>

<details><summary>힌트 3</summary>

Data(x=x, edge_index=edge_index)

</details>

원본: PracticeCode_2.ipynb 셀 9


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
from torch_geometric.data import Data
assert 'simple_graph' in globals() and isinstance(simple_graph, Data), "simple_graph = Data(x=x, edge_index=edge_index) 가 있어야 해요"
assert simple_graph.num_nodes == 3, f"노드는 3개여야 해요. 지금은 {simple_graph.num_nodes}"
assert simple_graph.num_edges == 4, f"엣지는 양방향 4줄이어야 해요. 지금은 {simple_graph.num_edges}"
assert simple_graph.edge_index.tolist() == [[0, 1, 1, 2], [1, 0, 2, 1]], "edge_index 는 [[0, 1, 1, 2], [1, 0, 2, 1]] 이어야 해요"
assert simple_graph.edge_index.dtype == torch.long, "edge_index 는 dtype=torch.long 이어야 해요"
assert simple_graph.x.dtype == torch.float and tuple(simple_graph.x.shape) == (3, 1), "x 는 (3, 1) 실수 표여야 해요"

print("통과! 원본 셀 9 예요. 셀 11 의 가라테 클럽 data 도 똑같은 Data 상자예요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.data import Data

edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
x = torch.tensor([[-1.0], [0.0], [1.0]], dtype=torch.float)

simple_graph = Data(x=x, edge_index=edge_index)
print(simple_graph)
print("Number of nodes:", simple_graph.num_nodes)
print("Number of edges:", simple_graph.num_edges)
```

</details>

## 2. 정답 y 와 훈련 표시 train_mask 달기  (빈칸 채우기)

8명 그래프 Data 에 정답 반 y 를 달고, 0번과 7번만 답을 아는 노드로 표시해요.

- 원본은 PyG 의 KarateClub() (34명)을 쓰지만, 채점을 빠르게 하려고 8명짜리 가짜 그래프(0~3번 한 무리, 4~7번 한 무리, 3번과 4번이 다리)로 바꿨어요. 내려받는 데이터는 없어요.
- `torch.tensor(edges).t()` 는 (11, 2) 표를 뒤집어 (2, 11) 로 만들어요. `to_undirected` 는 거꾸로 방향을 더해 양방향 22열로 만들어요. 원본 KarateClub 은 이미 양방향이라 이 두 줄이 없어요.
- `data.y = ...` 처럼 Data 상자에 새 이름표를 바로 달 수 있어요 (c05 의 속성 달기와 같아요).
- `torch.zeros(8, dtype=torch.bool)` 은 False 8개예요. `data.train_mask[0] = True` 로 0번만 True 로 바꿔요. True 인 노드만 학습 때 답을 봐요 (c11).
- 원본은 0번(Mr. Hi 관장)과 33번(사범), 여기서는 0번과 7번이 양쪽 대표예요.

<details><summary>힌트 1</summary>

정답 반은 정수라 torch.long

</details>

<details><summary>힌트 2</summary>

False 로 채우는 함수는 torch.zeros, dtype 은 torch.bool

</details>

<details><summary>힌트 3</summary>

마지막 노드 번호는 7

</details>

원본: PracticeCode_2.ipynb 셀 11


In [ ]:
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.___)
num_classes = 2

data.train_mask = torch.___(data.num_nodes, dtype=torch.___)
data.train_mask[0] = True
data.train_mask[___] = True

print(data)
print(f"Train mask (labeled ids): {data.train_mask.nonzero(as_tuple=False).view(-1).tolist()}")


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
assert 'data' in globals() and isinstance(data, Data), "data 는 Data 여야 해요"
assert data.num_edges == 22, f"양방향 엣지 22개여야 해요. 지금은 {data.num_edges}"
assert hasattr(data, 'y') and data.y is not None and data.y.tolist() == [0, 0, 0, 0, 1, 1, 1, 1], "data.y 는 [0, 0, 0, 0, 1, 1, 1, 1] 이어야 해요"
assert data.y.dtype == torch.long, "정답 반은 dtype=torch.long 이어야 해요. cross_entropy 가 정수 정답을 받아요"
assert hasattr(data, 'train_mask') and data.train_mask is not None and data.train_mask.dtype == torch.bool, "train_mask 는 dtype=torch.bool 이어야 해요"
assert data.train_mask.tolist() == [True, False, False, False, False, False, False, True], f"0번과 7번만 True 여야 해요. 지금은 {data.train_mask.tolist()}"

print("통과! 원본 셀 11 이에요. 이 data 하나를 뒤 실습 모델과 학습 함수에 그대로 넣어요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

print(data)
print(f"Train mask (labeled ids): {data.train_mask.nonzero(as_tuple=False).view(-1).tolist()}")

```

</details>

## 3. GCNConv 한 층 불러 쓰기  (따라 치기)

PyG 가 만들어 둔 GCN 층 GCNConv 를 하나 만들어 01 의 3명짜리 그래프에 한 번 통과시켜요. 노드마다 1칸이던 특징이 4칸이 돼요.

- w2-scratchgcn 에서 직접 짠 GCNLayer 를 PyG 는 `GCNConv` 한 줄로 줘요. 자기 자신 고리, 차수 정규화, 가중치 곱을 속에서 다 해요.
- `GCNConv(1, 4)` 는 입력 1칸, 출력 4칸. 부를 때는 `conv(x, edge_index)` 로 A_hat 대신 edge_index 를 넣어요.
- GCNConv 는 bias(더하는 숫자)가 있어서 식은 $\hat{A} X W^T + b$ 예요. 채점은 이 식으로 직접 계산해서 비교해요.
- 원본 셀 67 은 가라테 클럽 34칸 특징에 이 층을 써요. 여기서는 내려받는 것 없이 01 의 작은 그래프를 다시 써요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
x = torch.tensor([[-1.0], [0.0], [1.0]], dtype=torch.float)
simple_graph = Data(x=x, edge_index=edge_index)

torch.manual_seed(0)
conv = GCNConv(1, 4)
out = conv(simple_graph.x, simple_graph.edge_index)
print(out.shape)
```

<details><summary>힌트 1</summary>

from torch_geometric.nn import GCNConv

</details>

<details><summary>힌트 2</summary>

conv = GCNConv(1, 4)

</details>

<details><summary>힌트 3</summary>

out = conv(simple_graph.x, simple_graph.edge_index)

</details>

원본: PracticeCode_2.ipynb 셀 67


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
from torch_geometric.nn import GCNConv
assert 'conv' in globals() and isinstance(conv, GCNConv), "conv = GCNConv(1, 4) 가 있어야 해요"
assert tuple(conv.lin.weight.shape) == (4, 1), "GCNConv(1, 4) 로 입력 1칸, 출력 4칸이어야 해요"
assert 'out' in globals() and tuple(out.shape) == (3, 4), "out 은 (3, 4) 모양이어야 해요"
_At = torch.tensor([[1.0, 1.0, 0.0], [1.0, 1.0, 1.0], [0.0, 1.0, 1.0]])
_Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
_want = (_Dm @ _At @ _Dm) @ torch.tensor([[-1.0], [0.0], [1.0]]) @ conv.lin.weight.T + conv.bias
assert torch.allclose(out, _want, atol=1e-5), "out 이 정규화 식 A_hat X W^T + b 와 달라요. conv(simple_graph.x, simple_graph.edge_index) 로 불렀나요?"

print("통과! 원본 셀 67 의 PyGGCN 은 이 층 두 개를 쌓아요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
x = torch.tensor([[-1.0], [0.0], [1.0]], dtype=torch.float)
simple_graph = Data(x=x, edge_index=edge_index)

torch.manual_seed(0)
conv = GCNConv(1, 4)
out = conv(simple_graph.x, simple_graph.edge_index)
print(out.shape)
```

</details>

## 4. GCNConv 두 층 모델 PyGGCN  (빈칸 채우기)

GCNConv 두 개와 드롭아웃으로 모델을 만들어요. 순서는 층 1, ReLU, 드롭아웃, 층 2 예요.

- `nn.Dropout(p=0.5)` 는 학습 때 숨은 칸을 무작위로 절반쯤 0 으로 지워요. 몇몇 칸에만 기대지 않게 하는 장치예요. `model.eval()` 이면 지우지 않아요 (c11).
- `F.relu(h)` 는 음수를 0 으로 바꿔요 (c10).
- `return_hidden=True` 면 숨은 표현 h 와 점수 logits 를 **둘 다** 돌려줘요. 학습 중 그림(셀 64)에 h 를 저장하려고요.
- `data.num_node_features` 는 특징 칸 수, 여기서는 8 이에요.

<details><summary>힌트 1</summary>

conv1 은 (num_features, hidden_dim), conv2 는 (hidden_dim, num_classes)

</details>

<details><summary>힌트 2</summary>

relu 다음 dropout

</details>

<details><summary>힌트 3</summary>

층을 부를 때마다 edge_index 도 넘겨요

</details>

<details><summary>힌트 4</summary>

return h, logits

</details>

원본: PracticeCode_2.ipynb 셀 61, 67


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, ___)
        self.conv2 = GCNConv(___, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.___(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return ___
        return logits


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_model.eval()
logits = pyg_model(data.x, data.edge_index)
print(logits.shape)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
assert 'PyGGCN' in globals(), "PyGGCN 클래스를 만들어야 해요"
for _f, _c, _h in [(8, 2, 16), (5, 3, 4)]:
    try:
        torch.manual_seed(42)
        _m = PyGGCN(_f, _c, hidden_dim=_h)
    except Exception as _e:
        raise AssertionError(f"PyGGCN({_f}, {_c}, hidden_dim={_h}) 를 만들다가 에러가 났어요: {_e}")
    torch.manual_seed(42)
    _r = _RPyGGCN(_f, _c, hidden_dim=_h)
    for _k in ['conv1', 'conv2', 'dropout']:
        assert hasattr(_m, _k), f"self.{_k} 가 있어야 해요"
    assert isinstance(_m.conv1, GCNConv) and isinstance(_m.conv2, GCNConv), "conv1, conv2 는 GCNConv 층이어야 해요"
    assert tuple(_m.conv1.lin.weight.shape) == (_h, _f), f"conv1 은 GCNConv(num_features, hidden_dim) 이어야 해요. 지금 weight 모양은 {tuple(_m.conv1.lin.weight.shape)}"
    assert tuple(_m.conv2.lin.weight.shape) == (_c, _h), f"conv2 는 GCNConv(hidden_dim, num_classes) 이어야 해요. 지금 weight 모양은 {tuple(_m.conv2.lin.weight.shape)}"
    assert isinstance(_m.dropout, nn.Dropout) and abs(_m.dropout.p - 0.5) < 1e-9, "self.dropout = nn.Dropout(p=0.5) 여야 해요"
    torch.manual_seed(3)
    _xx = torch.randn(6, _f)
    _ei = to_undirected(torch.tensor([[0, 1, 2, 3, 4, 0], [1, 2, 3, 4, 5, 5]]))
    _m.eval()
    _r.eval()
    try:
        _out = _m(_xx, _ei)
        _hid = _m(_xx, _ei, return_hidden=True)
    except Exception as _e:
        raise AssertionError(f"모델을 부르다가 에러가 났어요: {_e}. forward 안에서 층마다 edge_index 를 넘겼는지 봐요")
    assert _out is not None and tuple(_out.shape) == (6, _c), f"model(x, edge_index) 는 (노드 수 6, 반 수 {_c}) 모양이어야 해요"
    assert torch.allclose(_out, _r(_xx, _ei), atol=1e-5), "점수가 원본과 달라요. conv1, relu, dropout, conv2 순서를 확인해요"
    assert isinstance(_hid, tuple) and len(_hid) == 2, "return_hidden=True 면 h, logits 두 값을 돌려줘야 해요"
    assert torch.allclose(_hid[0], _r(_xx, _ei, return_hidden=True)[0], atol=1e-5), "숨은 표현 h 가 원본과 달라요"

print("통과! 원본 셀 67 이에요. 셀 61 의 KarateClubGNN 도 GCNConv 자리에 CustomGCNConv 를 넣었을 뿐 모양이 같아요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_model.eval()
logits = pyg_model(data.x, data.edge_index)
print(logits.shape)

```

</details>

## 5. 둘째 층에 edge_index 를 안 넘겼어요  (고치기)

`forward() missing 1 required positional argument: 'edge_index'` 에러가 나요. 한 곳을 고쳐요.

- 에러 마지막 줄은 "forward 에 꼭 필요한 edge_index 가 빠졌다" 는 뜻이에요.
- GCNConv 는 누가 누구와 이웃인지 알아야 섞을 수 있어요. 그래서 층을 부를 **때마다** `(특징, edge_index)` 두 개를 넣어요. 일반 nn.Linear 는 하나만 넣어서 헷갈리기 쉬워요.

<details><summary>힌트 1</summary>

에러는 logits = ... 줄에서 나요

</details>

<details><summary>힌트 2</summary>

conv1 을 부르는 줄과 비교해 봐요

</details>

<details><summary>힌트 3</summary>

self.conv2(h, edge_index)

</details>

원본: PracticeCode_2.ipynb 셀 67


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h)
        if return_hidden:
            return h, logits
        return logits


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_model.eval()
logits = pyg_model(data.x, data.edge_index)
print(logits.shape)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
assert 'PyGGCN' in globals(), "PyGGCN 클래스를 만들어야 해요"
for _f, _c, _h in [(8, 2, 16), (5, 3, 4)]:
    try:
        torch.manual_seed(42)
        _m = PyGGCN(_f, _c, hidden_dim=_h)
    except Exception as _e:
        raise AssertionError(f"PyGGCN({_f}, {_c}, hidden_dim={_h}) 를 만들다가 에러가 났어요: {_e}")
    torch.manual_seed(42)
    _r = _RPyGGCN(_f, _c, hidden_dim=_h)
    for _k in ['conv1', 'conv2', 'dropout']:
        assert hasattr(_m, _k), f"self.{_k} 가 있어야 해요"
    assert isinstance(_m.conv1, GCNConv) and isinstance(_m.conv2, GCNConv), "conv1, conv2 는 GCNConv 층이어야 해요"
    assert tuple(_m.conv1.lin.weight.shape) == (_h, _f), f"conv1 은 GCNConv(num_features, hidden_dim) 이어야 해요. 지금 weight 모양은 {tuple(_m.conv1.lin.weight.shape)}"
    assert tuple(_m.conv2.lin.weight.shape) == (_c, _h), f"conv2 는 GCNConv(hidden_dim, num_classes) 이어야 해요. 지금 weight 모양은 {tuple(_m.conv2.lin.weight.shape)}"
    assert isinstance(_m.dropout, nn.Dropout) and abs(_m.dropout.p - 0.5) < 1e-9, "self.dropout = nn.Dropout(p=0.5) 여야 해요"
    torch.manual_seed(3)
    _xx = torch.randn(6, _f)
    _ei = to_undirected(torch.tensor([[0, 1, 2, 3, 4, 0], [1, 2, 3, 4, 5, 5]]))
    _m.eval()
    _r.eval()
    try:
        _out = _m(_xx, _ei)
        _hid = _m(_xx, _ei, return_hidden=True)
    except Exception as _e:
        raise AssertionError(f"모델을 부르다가 에러가 났어요: {_e}. forward 안에서 층마다 edge_index 를 넘겼는지 봐요")
    assert _out is not None and tuple(_out.shape) == (6, _c), f"model(x, edge_index) 는 (노드 수 6, 반 수 {_c}) 모양이어야 해요"
    assert torch.allclose(_out, _r(_xx, _ei), atol=1e-5), "점수가 원본과 달라요. conv1, relu, dropout, conv2 순서를 확인해요"
    assert isinstance(_hid, tuple) and len(_hid) == 2, "return_hidden=True 면 h, logits 두 값을 돌려줘야 해요"
    assert torch.allclose(_hid[0], _r(_xx, _ei, return_hidden=True)[0], atol=1e-5), "숨은 표현 h 가 원본과 달라요"

print("통과! Sample_Code_2 의 3층 모델(셀 41)도 층 셋 모두 edge_index 를 받아요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_model.eval()
logits = pyg_model(data.x, data.edge_index)
print(logits.shape)

```

</details>

## 6. 맞힌 개수로 정확도 세기  (따라 치기)

Sample_Code_2 방식으로 정확도를 구해요. 맞힌 개수를 세고 전체 수로 나눠요.

- `(pred == y).sum()` 은 True 를 1 로 세서 맞힌 개수예요. `int(...)` 로 텐서를 보통 정수로 바꿔요.
- `int(train_mask.sum())` 은 True 인 노드 수, 즉 훈련 노드 수예요.
- PracticeCode_2 의 `.float().mean()` 방식(w2-scratchgcn 06)과 결과는 같아요. 나누기 순서만 달라요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch

logits = torch.tensor([[2.0, 1.0], [0.1, 0.3], [1.5, -1.0], [0.0, 2.0], [0.2, 0.1]])
y = torch.tensor([0, 0, 0, 1, 1])
train_mask = torch.tensor([True, False, False, True, False])
pred = logits.argmax(dim=1)
train_correct = (pred[train_mask] == y[train_mask]).sum()
train_acc = int(train_correct) / int(train_mask.sum())
total_correct = (pred == y).sum()
total_acc = int(total_correct) / len(y)
```

<details><summary>힌트 1</summary>

pred = logits.argmax(dim=1)

</details>

<details><summary>힌트 2</summary>

맞힌 개수는 (비교).sum()

</details>

<details><summary>힌트 3</summary>

int(맞힌 개수) / int(전체 수)

</details>

원본: Sample_Code_2 (1).ipynb 셀 43


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
for _n in ['pred', 'train_acc', 'total_acc']:
    assert _n in globals(), f"{_n} 를 만들어야 해요"
assert pred.tolist() == [0, 1, 0, 1, 0], "pred 는 logits.argmax(dim=1) 이에요"
assert abs(train_acc - 1.0) < 1e-9, f"훈련 노드 2개 모두 맞혀서 1.0 이어야 해요. 지금은 {train_acc}"
assert abs(total_acc - 0.6) < 1e-9, f"5명 중 3명 맞혀서 0.6 이어야 해요. 지금은 {total_acc}"
assert isinstance(train_acc, float) and isinstance(total_acc, float), "int(...) / int(...) 로 보통 숫자를 만들어요"

print("통과! 원본 Sample_Code_2 셀 43 학습 반복 안의 정확도 네 줄이에요. 거기서 전체 수는 data.num_nodes 예요.")

<details><summary>정답 보기</summary>

```python
import torch

logits = torch.tensor([[2.0, 1.0], [0.1, 0.3], [1.5, -1.0], [0.0, 2.0], [0.2, 0.1]])
y = torch.tensor([0, 0, 0, 1, 1])
train_mask = torch.tensor([True, False, False, True, False])
pred = logits.argmax(dim=1)
train_correct = (pred[train_mask] == y[train_mask]).sum()
train_acc = int(train_correct) / int(train_mask.sum())
total_correct = (pred == y).sum()
total_acc = int(total_correct) / len(y)
```

</details>

## 7. 학습 함수 train_sparse_model  (빈칸 채우기)

Data 상자를 통째로 받아 학습하는 함수의 빈칸을 채워요. 바퀴마다 손실과 정확도를 기록하고, 정해진 바퀴에는 숨은 표현 사진(snapshot)을 저장해요.

- 원본은 PyG 의 KarateClub() (34명)을 쓰지만, 채점을 빠르게 하려고 8명짜리 가짜 그래프(0~3번 한 무리, 4~7번 한 무리, 3번과 4번이 다리)로 바꿨어요. 내려받는 데이터는 없어요. 바퀴 수도 원본 300 대신 60 으로 줄였어요.
- w2-scratchgcn 의 train_dense_model 과 같은 순서예요. 다른 점은 `data.x`, `data.edge_index`, `data.train_mask` 처럼 Data 상자에서 꺼내 쓰는 것뿐이에요.
- `if epoch in [0, 10, 50, ...]:` 은 목록 안에 있는 바퀴일 때만 저장해요. `range(epochs)` 라서 epoch 는 0 부터 시작해요.
- `hidden.detach().clone()` 은 계산 기록을 떼고(detach) 복사(clone)해요. 복사하지 않으면 나중 값으로 바뀔 수 있어요.

<details><summary>힌트 1</summary>

모델 입력은 data.x, data.edge_index

</details>

<details><summary>힌트 2</summary>

손실은 data.train_mask 인 노드만

</details>

<details><summary>힌트 3</summary>

평가 때 return_hidden=True, 저장은 hidden.detach().clone()

</details>

원본: PracticeCode_2.ipynb 셀 61, 67


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_sparse_model(model, data, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []
    snapshots = {}

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(data.___, data.___)
        loss = F.cross_entropy(logits[data.___], data.y[data.___])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            hidden, logits_eval = model(data.x, data.edge_index, return_hidden=___)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, data.y, data.train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch in [0, 10, 50, 100, 199, 299]:
            snapshots[epoch] = hidden.detach().___()

        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch {epoch + 1:3d}/{epochs} | Loss: {loss.item():.4f} | "
                f"Train Acc: {train_acc:.4f} | Total Acc: {full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full, snapshots


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_losses, pyg_train_accs, pyg_full_accs, pyg_snapshots = train_sparse_model(
    pyg_model, data, epochs=60, name="the PyG GCNConv model"
)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
for _n in ['pyg_losses', 'pyg_full_accs', 'pyg_snapshots']:
    assert _n in globals(), f"{_n} 기록이 있어야 해요"
assert len(pyg_losses) == 60, f"60바퀴 손실이 모여야 해요. 지금은 {len(pyg_losses)}개예요"
torch.manual_seed(42)
_rm = _RPyGGCN(8, 2)
_rl, _rt, _rf, _rs = _r_train(_rm, _D8, 60, 0.01)
assert abs(pyg_losses[0] - _rl[0]) < 1e-5, f"첫 손실이 원본 {_rl[0]:.4f} 와 달라요(지금 {pyg_losses[0]:.4f}). data.train_mask 노드만 채점했는지 봐요"
assert max(abs(a - b) for a, b in zip(pyg_losses, _rl)) < 1e-4, "손실 기록이 원본과 달라요. zero_grad, backward, step 과 model.eval() 위치를 확인해요"
assert pyg_full_accs == _rf, "전체 정확도 기록이 원본과 달라요. 평가는 model.eval() 과 torch.no_grad() 안에서 해요"
assert sorted(pyg_snapshots.keys()) == [0, 10, 50], f"60바퀴면 0, 10, 50 번째 바퀴 숨은 표현이 저장돼야 해요. 지금은 {sorted(pyg_snapshots.keys())}"
assert torch.allclose(pyg_snapshots[50], _rs[50], atol=1e-5), "50 번째 바퀴 숨은 표현이 원본과 달라요"

print("통과! 원본 셀 61 이에요. 노트북은 이 함수로 CustomGCNConv 모델과 PyGGCN 을 똑같이 학습해 비교해요(셀 67, 68).")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_sparse_model(model, data, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []
    snapshots = {}

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            hidden, logits_eval = model(data.x, data.edge_index, return_hidden=True)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, data.y, data.train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch in [0, 10, 50, 100, 199, 299]:
            snapshots[epoch] = hidden.detach().clone()

        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch {epoch + 1:3d}/{epochs} | Loss: {loss.item():.4f} | "
                f"Train Acc: {train_acc:.4f} | Total Acc: {full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full, snapshots


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_losses, pyg_train_accs, pyg_full_accs, pyg_snapshots = train_sparse_model(
    pyg_model, data, epochs=60, name="the PyG GCNConv model"
)

```

</details>

## 8. 평가할 때도 드롭아웃이 켜져 있어요  (고치기)

에러는 없는데 정확도 기록이 들쭉날쭉하고 원본과 달라요. 평가 앞에 빠진 한 줄을 넣어요.

- `model.train()` 상태에서는 드롭아웃이 칸을 무작위로 지워요. 채점(평가)할 때까지 지우면 같은 모델인데 점수가 매번 달라져요.
- `torch.no_grad()` 는 기울기 계산만 끄고, 드롭아웃은 끄지 않아요. 드롭아웃을 끄는 건 `model.eval()` 이에요 (c11-10).
- 에러가 안 나는 실수라서 더 조심해야 해요. 학습 곡선이 이상하게 흔들리면 먼저 의심해요.

<details><summary>힌트 1</summary>

with torch.no_grad(): 바로 위를 봐요

</details>

<details><summary>힌트 2</summary>

평가 모드로 바꾸는 줄이 없어요

</details>

<details><summary>힌트 3</summary>

model.eval()

</details>

원본: PracticeCode_2.ipynb 셀 61


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_sparse_model(model, data, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []
    snapshots = {}

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            hidden, logits_eval = model(data.x, data.edge_index, return_hidden=True)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, data.y, data.train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch in [0, 10, 50, 100, 199, 299]:
            snapshots[epoch] = hidden.detach().clone()

        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch {epoch + 1:3d}/{epochs} | Loss: {loss.item():.4f} | "
                f"Train Acc: {train_acc:.4f} | Total Acc: {full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full, snapshots


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_losses, pyg_train_accs, pyg_full_accs, pyg_snapshots = train_sparse_model(
    pyg_model, data, epochs=60, name="the PyG GCNConv model"
)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
for _n in ['pyg_losses', 'pyg_full_accs', 'pyg_snapshots']:
    assert _n in globals(), f"{_n} 기록이 있어야 해요"
assert len(pyg_losses) == 60, f"60바퀴 손실이 모여야 해요. 지금은 {len(pyg_losses)}개예요"
torch.manual_seed(42)
_rm = _RPyGGCN(8, 2)
_rl, _rt, _rf, _rs = _r_train(_rm, _D8, 60, 0.01)
assert abs(pyg_losses[0] - _rl[0]) < 1e-5, f"첫 손실이 원본 {_rl[0]:.4f} 와 달라요(지금 {pyg_losses[0]:.4f}). data.train_mask 노드만 채점했는지 봐요"
assert max(abs(a - b) for a, b in zip(pyg_losses, _rl)) < 1e-4, "손실 기록이 원본과 달라요. zero_grad, backward, step 과 model.eval() 위치를 확인해요"
assert pyg_full_accs == _rf, "전체 정확도 기록이 원본과 달라요. 평가는 model.eval() 과 torch.no_grad() 안에서 해요"
assert sorted(pyg_snapshots.keys()) == [0, 10, 50], f"60바퀴면 0, 10, 50 번째 바퀴 숨은 표현이 저장돼야 해요. 지금은 {sorted(pyg_snapshots.keys())}"
assert torch.allclose(pyg_snapshots[50], _rs[50], atol=1e-5), "50 번째 바퀴 숨은 표현이 원본과 달라요"

print("통과! 원본 셀 61, 67 모두 평가 앞에 model.eval(), 다음 바퀴 앞에 model.train() 을 불러요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_sparse_model(model, data, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []
    snapshots = {}

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            hidden, logits_eval = model(data.x, data.edge_index, return_hidden=True)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, data.y, data.train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch in [0, 10, 50, 100, 199, 299]:
            snapshots[epoch] = hidden.detach().clone()

        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch {epoch + 1:3d}/{epochs} | Loss: {loss.item():.4f} | "
                f"Train Acc: {train_acc:.4f} | Total Acc: {full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full, snapshots


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_losses, pyg_train_accs, pyg_full_accs, pyg_snapshots = train_sparse_model(
    pyg_model, data, epochs=60, name="the PyG GCNConv model"
)

```

</details>

## 9. MessagePassing 으로 만든 GCN 층  (빈칸 채우기)

PyG 의 기본 틀 MessagePassing 을 물려받아 GCN 층을 만들어요. forward 는 준비를, message 는 이웃 하나가 보낼 메시지를 정해요.

- `super().__init__(aggr="add")` 는 부모 틀에게 "메시지를 합으로 모아" 라고 알려요. 모으기(Aggregate)는 부모가 해 줘요 (c06 상속).
- `self.propagate(edge_index, x=x, norm=norm)` 를 부르면 부모가 엣지마다 `message(x_j, norm)` 을 불러요. `x_j` 는 그 엣지 **출발 노드**의 특징이에요.
- `norm.view(-1, 1)` 은 길이 E 인 계수를 (E, 1) 세로 막대로 바꿔서 (E, 칸 수) 표와 곱할 수 있게 해요 (c09).
- `nn.Parameter(torch.zeros(out_channels))` 는 학습으로 바뀌는 숫자 묶음을 직접 만드는 방법이에요. 여기서는 bias 예요.
- 채점은 같은 가중치를 넣은 PyG GCNConv 와 결과가 똑같은지 봐요.

<details><summary>힌트 1</summary>

모으기는 합, 영어로 add

</details>

<details><summary>힌트 2</summary>

자기 자신 고리 함수는 add_self_loops

</details>

<details><summary>힌트 3</summary>

1/루트는 pow(-0.5)

</details>

<details><summary>힌트 4</summary>

메시지는 norm.view(-1, 1) * x_j

</details>

원본: PracticeCode_2.ipynb 셀 60 / Sample_Code_2 (1).ipynb 셀 38


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

class CustomGCNConv(MessagePassing):
    '''
    GCN layer on top of PyG MessagePassing.

    h_v = W * SUM_{u in N(v) U {v}} (1/sqrt(deg(v) deg(u))) * h_u
    '''

    def __init__(self, in_channels, out_channels):
        super().__init__(aggr="___")
        self.lin = nn.Linear(in_channels, out_channels, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # Add self-loops so each node also keeps its own features
        edge_index, _ = ___(edge_index, num_nodes=x.size(0))

        # Shared linear transform, then degree-normalized messages
        x = self.lin(x)
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(___)
        deg_inv_sqrt[deg_inv_sqrt == float("inf")] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        out = self.propagate(edge_index, x=x, norm=norm)
        return out + self.bias

    def message(self, x_j, norm):
        # x_j is the source-node feature on each edge
        return norm.view(-1, 1) * ___


torch.manual_seed(0)
layer = CustomGCNConv(8, 3)
x_in = torch.randn(8, 8)
edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True
out = layer(x_in, data.edge_index)
print(out.shape)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
from torch_geometric.nn import MessagePassing
assert 'CustomGCNConv' in globals(), "CustomGCNConv 클래스가 있어야 해요"
try:
    torch.manual_seed(0)
    _c = CustomGCNConv(8, 3)
except Exception as _e:
    raise AssertionError(f"CustomGCNConv(8, 3) 을 만들다가 에러가 났어요: {_e}")
assert isinstance(_c, MessagePassing), "CustomGCNConv 는 MessagePassing 을 물려받아야 해요"
assert getattr(_c, 'aggr', None) == 'add', f"모으기 방법은 aggr=\"add\" (합) 이어야 해요. 지금은 {getattr(_c, 'aggr', None)}"
assert hasattr(_c, 'lin') and tuple(_c.lin.weight.shape) == (3, 8) and _c.lin.bias is None, "self.lin = nn.Linear(in_channels, out_channels, bias=False) 여야 해요"
assert hasattr(_c, 'bias') and isinstance(_c.bias, nn.Parameter) and torch.all(_c.bias == 0), "self.bias 는 0 으로 시작하는 nn.Parameter 여야 해요"
_g = GCNConv(8, 3)
with torch.no_grad():
    _g.lin.weight.copy_(_c.lin.weight)
    _c.bias.copy_(torch.tensor([0.1, -0.2, 0.3]))
    _g.bias.copy_(_c.bias)
for _ei in [_D8.edge_index, to_undirected(torch.tensor([[0, 1, 2, 3, 4, 0], [1, 2, 3, 4, 5, 5]]))]:
    _n = int(_ei.max()) + 1
    torch.manual_seed(4)
    _xx = torch.randn(_n, 8)
    try:
        _o = _c(_xx, _ei)
    except Exception as _e:
        raise AssertionError(f"층을 부르다가 에러가 났어요: {_e}")
    assert _o is not None and tuple(_o.shape) == (_n, 3), f"결과는 ({_n}, 3) 모양이어야 해요"
    assert torch.allclose(_o, _g(_xx, _ei), atol=1e-5), "같은 가중치의 PyG GCNConv 결과와 달라요. 자기 자신 고리, pow(-0.5), norm.view(-1, 1) * x_j 를 확인해요"

print("통과! 원본 셀 60 이에요. 셀 61 의 KarateClubGNN 이 이 층 두 개를 쌓아요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

class CustomGCNConv(MessagePassing):
    '''
    GCN layer on top of PyG MessagePassing.

    h_v = W * SUM_{u in N(v) U {v}} (1/sqrt(deg(v) deg(u))) * h_u
    '''

    def __init__(self, in_channels, out_channels):
        super().__init__(aggr="add")
        self.lin = nn.Linear(in_channels, out_channels, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # Add self-loops so each node also keeps its own features
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Shared linear transform, then degree-normalized messages
        x = self.lin(x)
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float("inf")] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        out = self.propagate(edge_index, x=x, norm=norm)
        return out + self.bias

    def message(self, x_j, norm):
        # x_j is the source-node feature on each edge
        return norm.view(-1, 1) * x_j


torch.manual_seed(0)
layer = CustomGCNConv(8, 3)
x_in = torch.randn(8, 8)
edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True
out = layer(x_in, data.edge_index)
print(out.shape)

```

</details>

## 10. PyGGCN 직접 짜기  (직접 짜기)

주석만 보고 GCNConv 두 층 모델 PyGGCN 을 짜요.

- 04 를 보지 않고 다시 써 봐요. 층을 만드는 순서(conv1, conv2, dropout)가 원본과 같아야 seed 가 같을 때 가중치도 같아요.
- 채점은 크기가 다른 두 모델을 원본 코드와 비교해요.

<details><summary>힌트 1</summary>

class PyGGCN(nn.Module): 와 super().__init__()

</details>

<details><summary>힌트 2</summary>

conv1, conv2, dropout 세 속성

</details>

<details><summary>힌트 3</summary>

forward: conv1, relu, dropout, conv2

</details>

<details><summary>힌트 4</summary>

return_hidden 이면 return h, logits

</details>

원본: PracticeCode_2.ipynb 셀 67


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

# PyGGCN(nn.Module)
# 1. __init__(self, num_features, num_classes, hidden_dim=16):
#    부모 준비, self.conv1 = GCNConv(num_features, hidden_dim),
#    self.conv2 = GCNConv(hidden_dim, num_classes), self.dropout = nn.Dropout(p=0.5)
# 2. forward(self, x, edge_index, return_hidden=False):
#    h = conv1(x, edge_index) 다음 F.relu, 다음 dropout
#    logits = conv2(h, edge_index)
#    return_hidden 이면 h, logits 둘 다, 아니면 logits 를 돌려줘요
class PyGGCN(nn.Module):
    ...


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
assert 'PyGGCN' in globals(), "PyGGCN 클래스를 만들어야 해요"
for _f, _c, _h in [(8, 2, 16), (5, 3, 4)]:
    try:
        torch.manual_seed(42)
        _m = PyGGCN(_f, _c, hidden_dim=_h)
    except Exception as _e:
        raise AssertionError(f"PyGGCN({_f}, {_c}, hidden_dim={_h}) 를 만들다가 에러가 났어요: {_e}")
    torch.manual_seed(42)
    _r = _RPyGGCN(_f, _c, hidden_dim=_h)
    for _k in ['conv1', 'conv2', 'dropout']:
        assert hasattr(_m, _k), f"self.{_k} 가 있어야 해요"
    assert isinstance(_m.conv1, GCNConv) and isinstance(_m.conv2, GCNConv), "conv1, conv2 는 GCNConv 층이어야 해요"
    assert tuple(_m.conv1.lin.weight.shape) == (_h, _f), f"conv1 은 GCNConv(num_features, hidden_dim) 이어야 해요. 지금 weight 모양은 {tuple(_m.conv1.lin.weight.shape)}"
    assert tuple(_m.conv2.lin.weight.shape) == (_c, _h), f"conv2 는 GCNConv(hidden_dim, num_classes) 이어야 해요. 지금 weight 모양은 {tuple(_m.conv2.lin.weight.shape)}"
    assert isinstance(_m.dropout, nn.Dropout) and abs(_m.dropout.p - 0.5) < 1e-9, "self.dropout = nn.Dropout(p=0.5) 여야 해요"
    torch.manual_seed(3)
    _xx = torch.randn(6, _f)
    _ei = to_undirected(torch.tensor([[0, 1, 2, 3, 4, 0], [1, 2, 3, 4, 5, 5]]))
    _m.eval()
    _r.eval()
    try:
        _out = _m(_xx, _ei)
        _hid = _m(_xx, _ei, return_hidden=True)
    except Exception as _e:
        raise AssertionError(f"모델을 부르다가 에러가 났어요: {_e}. forward 안에서 층마다 edge_index 를 넘겼는지 봐요")
    assert _out is not None and tuple(_out.shape) == (6, _c), f"model(x, edge_index) 는 (노드 수 6, 반 수 {_c}) 모양이어야 해요"
    assert torch.allclose(_out, _r(_xx, _ei), atol=1e-5), "점수가 원본과 달라요. conv1, relu, dropout, conv2 순서를 확인해요"
    assert isinstance(_hid, tuple) and len(_hid) == 2, "return_hidden=True 면 h, logits 두 값을 돌려줘야 해요"
    assert torch.allclose(_hid[0], _r(_xx, _ei, return_hidden=True)[0], atol=1e-5), "숨은 표현 h 가 원본과 달라요"

print("통과! 셀 67 을 혼자 썼어요. GCNConv 를 CustomGCNConv 로 바꾸면 셀 61 의 KarateClubGNN 이 돼요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

```

</details>

## 11. train_sparse_model 직접 짜기  (직접 짜기)

주석만 보고 학습 함수 train_sparse_model 을 짜요. 모델, 정확도 함수, data 는 이미 들어 있어요.

- 07 에서 빈칸으로 채운 함수를 통째로 써요. print 줄은 넣어도 되고 빼도 채점과 상관없어요.
- 채점은 60바퀴 손실, 정확도, 저장한 숨은 표현이 원본과 같은지 봐요. 그리고 다른 lr 에서도 같은지 봐요.
- 평가 앞 `model.eval()` 을 잊지 마세요 (08).

<details><summary>힌트 1</summary>

옵티마이저와 기록 리스트, snapshots = {} 는 반복 밖

</details>

<details><summary>힌트 2</summary>

for epoch in range(epochs): 안에서 train 여섯 줄

</details>

<details><summary>힌트 3</summary>

eval, no_grad 안에서 hidden, logits_eval 과 accuracy_from_logits

</details>

<details><summary>힌트 4</summary>

기록 append, 정해진 epoch 이면 snapshot, 반복 뒤 return 네 값

</details>

원본: PracticeCode_2.ipynb 셀 61


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


# train_sparse_model(model, data, epochs=300, lr=0.01, name="model")
# 1. optimizer = Adam(model.parameters(), lr=lr, weight_decay=5e-4)
# 2. history_loss, history_train, history_full = 빈 리스트 세 개, snapshots = 빈 딕셔너리
# 3. epoch 를 0 부터 epochs - 1 까지 돌면서
#    3-1. model.train(), optimizer.zero_grad()
#    3-2. logits = model(data.x, data.edge_index)
#    3-3. loss = data.train_mask 노드만 F.cross_entropy, loss.backward(), optimizer.step()
#    3-4. model.eval() 하고 with torch.no_grad(): 안에서
#         hidden, logits_eval = model(data.x, data.edge_index, return_hidden=True)
#         accuracy_from_logits 로 train_acc, full_acc
#    3-5. 세 리스트에 loss.item(), train_acc, full_acc 를 붙여요
#    3-6. epoch 가 [0, 10, 50, 100, 199, 299] 안에 있으면 snapshots[epoch] = hidden.detach().clone()
# 4. history_loss, history_train, history_full, snapshots 를 돌려줘요
def train_sparse_model(model, data, epochs=300, lr=0.01, name="model"):
    ...


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_losses, pyg_train_accs, pyg_full_accs, pyg_snapshots = train_sparse_model(
    pyg_model, data, epochs=60, name="the PyG GCNConv model"
)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
for _n in ['pyg_losses', 'pyg_full_accs', 'pyg_snapshots']:
    assert _n in globals(), f"{_n} 기록이 있어야 해요"
assert len(pyg_losses) == 60, f"60바퀴 손실이 모여야 해요. 지금은 {len(pyg_losses)}개예요"
torch.manual_seed(42)
_rm = _RPyGGCN(8, 2)
_rl, _rt, _rf, _rs = _r_train(_rm, _D8, 60, 0.01)
assert abs(pyg_losses[0] - _rl[0]) < 1e-5, f"첫 손실이 원본 {_rl[0]:.4f} 와 달라요(지금 {pyg_losses[0]:.4f}). data.train_mask 노드만 채점했는지 봐요"
assert max(abs(a - b) for a, b in zip(pyg_losses, _rl)) < 1e-4, "손실 기록이 원본과 달라요. zero_grad, backward, step 과 model.eval() 위치를 확인해요"
assert pyg_full_accs == _rf, "전체 정확도 기록이 원본과 달라요. 평가는 model.eval() 과 torch.no_grad() 안에서 해요"
assert sorted(pyg_snapshots.keys()) == [0, 10, 50], f"60바퀴면 0, 10, 50 번째 바퀴 숨은 표현이 저장돼야 해요. 지금은 {sorted(pyg_snapshots.keys())}"
assert torch.allclose(pyg_snapshots[50], _rs[50], atol=1e-5), "50 번째 바퀴 숨은 표현이 원본과 달라요"
torch.manual_seed(7)
_m1 = PyGGCN(8, 2, hidden_dim=4)
_g = train_sparse_model(_m1, _D8, epochs=12, lr=0.05, name="t")
torch.manual_seed(7)
_m2 = _RPyGGCN(8, 2, hidden_dim=4)
_w = _r_train(_m2, _D8, 12, 0.05)
assert _g is not None and len(_g) == 4, "history_loss, history_train, history_full, snapshots 네 값을 돌려줘야 해요"
assert len(_g[0]) == 12 and max(abs(a - b) for a, b in zip(_g[0], _w[0])) < 1e-5, "epochs=12, lr=0.05 에서 손실이 원본과 달라요. lr=lr 과 weight_decay=5e-4 를 확인해요"
assert sorted(_g[3].keys()) == [0, 10], "12바퀴면 0, 10 번째 바퀴만 저장돼요"

print("통과! 셀 61 을 혼자 썼어요. 이제 2주차 노트북 모델 비교(셀 68, 70)를 스스로 돌릴 수 있어요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_sparse_model(model, data, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []
    snapshots = {}

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            hidden, logits_eval = model(data.x, data.edge_index, return_hidden=True)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, data.y, data.train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch in [0, 10, 50, 100, 199, 299]:
            snapshots[epoch] = hidden.detach().clone()

        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch {epoch + 1:3d}/{epochs} | Loss: {loss.item():.4f} | "
                f"Train Acc: {train_acc:.4f} | Total Acc: {full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full, snapshots


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
edge_index = to_undirected(torch.tensor(edges, dtype=torch.long).t())
data = Data(x=torch.eye(8), edge_index=edge_index)

data.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
num_classes = 2

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[7] = True

torch.manual_seed(42)
pyg_model = PyGGCN(data.num_node_features, num_classes)
pyg_losses, pyg_train_accs, pyg_full_accs, pyg_snapshots = train_sparse_model(
    pyg_model, data, epochs=60, name="the PyG GCNConv model"
)

```

</details>

## 12. CustomGCNConv 직접 짜기  (직접 짜기)

주석만 보고 MessagePassing 을 물려받은 GCN 층 CustomGCNConv 를 짜요. PyG 의 GCNConv 와 같은 값이 나와야 해요.

- 09 의 빈칸 네 곳만이 아니라 클래스 전체를 써요. 식은 $h_v = W \sum_{u \in N(v) \cup \{v\}} \frac{1}{\sqrt{\deg(v)\deg(u)}} h_u + b$ 예요.
- forward 순서: 자기 자신 고리 붙이기 → `self.lin(x)` → 차수와 계수 norm → `self.propagate` → bias 더하기.
- 채점은 서로 다른 그래프 두 개에서 같은 가중치의 GCNConv 와 비교해요.

<details><summary>힌트 1</summary>

__init__: super().__init__(aggr="add"), self.lin, self.bias, self.reset_parameters()

</details>

<details><summary>힌트 2</summary>

reset_parameters: lin.reset_parameters(), bias.data.zero_()

</details>

<details><summary>힌트 3</summary>

forward: add_self_loops, lin, degree(col, ...), pow(-0.5), inf 는 0, norm

</details>

<details><summary>힌트 4</summary>

propagate 결과 + self.bias, message 는 norm.view(-1, 1) * x_j

</details>

원본: PracticeCode_2.ipynb 셀 60 / Sample_Code_2 (1).ipynb 셀 38


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

# CustomGCNConv(MessagePassing)
# 1. __init__(self, in_channels, out_channels):
#    부모 준비는 super().__init__(aggr="add")
#    self.lin = bias 없는 nn.Linear(in_channels, out_channels)
#    self.bias = nn.Parameter(torch.zeros(out_channels)), 그다음 self.reset_parameters()
# 2. reset_parameters(self): self.lin.reset_parameters(), self.bias.data.zero_()
# 3. forward(self, x, edge_index):
#    3-1. edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
#    3-2. x = self.lin(x), row, col = edge_index
#    3-3. deg = degree(col, x.size(0), dtype=x.dtype), deg_inv_sqrt = deg 의 -0.5 제곱, inf 는 0
#    3-4. norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
#    3-5. out = self.propagate(edge_index, x=x, norm=norm), out + self.bias 를 돌려줘요
# 4. message(self, x_j, norm): norm 을 (-1, 1) 모양으로 바꿔 x_j 와 곱해 돌려줘요
class CustomGCNConv(MessagePassing):
    ...


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

def _r_data():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _d = Data(x=torch.eye(8), edge_index=to_undirected(torch.tensor(_e, dtype=torch.long).t()))
    _d.y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)
    _d.train_mask = torch.zeros(8, dtype=torch.bool)
    _d.train_mask[0] = True
    _d.train_mask[7] = True
    return _d

class _RPyGGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

def _r_train(model, data, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf, snaps = [], [], [], {}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        l = F.cross_entropy(model(data.x, data.edge_index)[data.train_mask], data.y[data.train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            hid, lg = model(data.x, data.edge_index, return_hidden=True)
            p = lg.argmax(dim=1)
            ht.append((p[data.train_mask] == data.y[data.train_mask]).float().mean().item())
            hf.append((p == data.y).float().mean().item())
        hl.append(l.item())
        if ep in [0, 10, 50, 100, 199, 299]:
            snaps[ep] = hid.detach().clone()
    return hl, ht, hf, snaps

_D8 = _r_data()
from torch_geometric.nn import MessagePassing
assert 'CustomGCNConv' in globals(), "CustomGCNConv 클래스가 있어야 해요"
try:
    torch.manual_seed(0)
    _c = CustomGCNConv(8, 3)
except Exception as _e:
    raise AssertionError(f"CustomGCNConv(8, 3) 을 만들다가 에러가 났어요: {_e}")
assert isinstance(_c, MessagePassing), "CustomGCNConv 는 MessagePassing 을 물려받아야 해요"
assert getattr(_c, 'aggr', None) == 'add', f"모으기 방법은 aggr=\"add\" (합) 이어야 해요. 지금은 {getattr(_c, 'aggr', None)}"
assert hasattr(_c, 'lin') and tuple(_c.lin.weight.shape) == (3, 8) and _c.lin.bias is None, "self.lin = nn.Linear(in_channels, out_channels, bias=False) 여야 해요"
assert hasattr(_c, 'bias') and isinstance(_c.bias, nn.Parameter) and torch.all(_c.bias == 0), "self.bias 는 0 으로 시작하는 nn.Parameter 여야 해요"
_g = GCNConv(8, 3)
with torch.no_grad():
    _g.lin.weight.copy_(_c.lin.weight)
    _c.bias.copy_(torch.tensor([0.1, -0.2, 0.3]))
    _g.bias.copy_(_c.bias)
for _ei in [_D8.edge_index, to_undirected(torch.tensor([[0, 1, 2, 3, 4, 0], [1, 2, 3, 4, 5, 5]]))]:
    _n = int(_ei.max()) + 1
    torch.manual_seed(4)
    _xx = torch.randn(_n, 8)
    try:
        _o = _c(_xx, _ei)
    except Exception as _e:
        raise AssertionError(f"층을 부르다가 에러가 났어요: {_e}")
    assert _o is not None and tuple(_o.shape) == (_n, 3), f"결과는 ({_n}, 3) 모양이어야 해요"
    assert torch.allclose(_o, _g(_xx, _ei), atol=1e-5), "같은 가중치의 PyG GCNConv 결과와 달라요. 자기 자신 고리, pow(-0.5), norm.view(-1, 1) * x_j 를 확인해요"

print("통과! PyG 의 GCNConv, 그리고 3주차 GraphSAGE, GIN 층도 전부 이 MessagePassing 틀로 만들어져 있어요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

class CustomGCNConv(MessagePassing):
    '''
    GCN layer on top of PyG MessagePassing.

    h_v = W * SUM_{u in N(v) U {v}} (1/sqrt(deg(v) deg(u))) * h_u
    '''

    def __init__(self, in_channels, out_channels):
        super().__init__(aggr="add")
        self.lin = nn.Linear(in_channels, out_channels, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # Add self-loops so each node also keeps its own features
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Shared linear transform, then degree-normalized messages
        x = self.lin(x)
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float("inf")] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        out = self.propagate(edge_index, x=x, norm=norm)
        return out + self.bias

    def message(self, x_j, norm):
        # x_j is the source-node feature on each edge
        return norm.view(-1, 1) * x_j

```

</details>